In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score
)

import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
np.random.seed(42)

n = 1000

locations = ['Hostel', 'Market', 'Residential', 'College', 'Hospital']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

data = []

for i in range(n):
    location = np.random.choice(locations)
    day = np.random.choice(days)

    temperature = np.random.randint(20, 41)

    # Different locations have different waste generation rates
    population_map = {
        'Hostel': 500,
        'Market': 1200,
        'Residential': 800,
        'College': 1500,
        'Hospital': 1000
    }

    population = population_map[location]

    # Weekend waste generation is generally higher
    weekend_factor = 1.2 if day in ['Saturday', 'Sunday'] else 1.0

    # Location-based waste generation
    location_factor = {
        'Hostel': 0.6,
        'Market': 0.9,
        'Residential': 0.5,
        'College': 0.7,
        'Hospital': 0.8
    }

    base_waste = location_factor[location] * 70

    waste_level = (
        base_waste * weekend_factor
        + temperature * 0.3
        + np.random.normal(0, 8)
    )

    waste_level = np.clip(waste_level, 0, 100)

    data.append([
        f'B{i+1}',
        location,
        day,
        temperature,
        population,
        round(waste_level, 2)
    ])

df = pd.DataFrame(data, columns=[
    'Bin_ID',
    'Location',
    'Day',
    'Temperature',
    'Population',
    'Waste_Level'
])

df.head()

In [ ]:
df.to_csv('waste_management.csv', index=False)

print("Dataset saved successfully!")
print("Total Records:", len(df))

In [ ]:
print("First 5 Records:")
display(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
print("Duplicate Records:", df.duplicated().sum())

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df['Waste_Level'], bins=20, kde=True)

plt.title('Distribution of Garbage Waste Levels')
plt.xlabel('Waste Level (%)')
plt.ylabel('Number of Bins')

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=df,
    x='Location',
    y='Waste_Level',
    estimator=np.mean
)

plt.title('Average Waste Generation by Location')
plt.xlabel('Location')
plt.ylabel('Average Waste Level (%)')

plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    data=df,
    x='Day',
    y='Waste_Level',
    estimator=np.mean,
    order=days
)

plt.title('Average Waste Level Across Days')
plt.xlabel('Day')
plt.ylabel('Average Waste Level (%)')

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

numeric_df = df[
    ['Temperature', 'Population', 'Waste_Level']
]

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Between Dataset Attributes')

plt.show()

In [ ]:
def classify_waste(level):

    if level <= 40:
        return 'Low'

    elif level <= 75:
        return 'Medium'

    else:
        return 'High'


df['Waste_Category'] = df['Waste_Level'].apply(classify_waste)

df.head()

In [ ]:
df_encoded = df.copy()

le_location = LabelEncoder()
le_day = LabelEncoder()

df_encoded['Location'] = le_location.fit_transform(
    df_encoded['Location']
)

df_encoded['Day'] = le_day.fit_transform(
    df_encoded['Day']
)

print(df_encoded.head())

In [ ]:
X = df_encoded[['Location', 'Day', 'Temperature', 'Population']]

y = df_encoded['Waste_Category']

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

print("Random Forest Model Trained Successfully!")

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Low', 'Medium', 'High'])

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Low', 'Medium', 'High'],
    yticklabels=['Low', 'Medium', 'High']
)

plt.xlabel('Predicted Category')
plt.ylabel('Actual Category')
plt.title('Confusion Matrix')

plt.show()

In [ ]:
cluster_data = df[
    ['Temperature', 'Population', 'Waste_Level']
]

scaler = StandardScaler()

scaled_data = scaler.fit_transform(cluster_data)

print("Data prepared for clustering!")

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df['Cluster'] = kmeans.fit_predict(scaled_data)

print(df[['Bin_ID', 'Location', 'Waste_Level', 'Cluster']].head(10))

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df,
    x='Population',
    y='Waste_Level',
    hue='Cluster',
    palette='viridis',
    s=70
)

plt.title('Garbage Bins Grouped Using K-Means')
plt.xlabel('Population')
plt.ylabel('Waste Level (%)')

plt.show()

In [ ]:
cluster_summary = df.groupby('Cluster').agg(
    Average_Waste=('Waste_Level', 'mean'),
    Average_Population=('Population', 'mean'),
    Total_Bins=('Bin_ID', 'count')
)

print(cluster_summary)

In [ ]:
score = silhouette_score(scaled_data, df['Cluster'])

print("Silhouette Score:", round(score, 3))

In [ ]:
def collection_priority(category):

    if category == 'High':
        return 'URGENT COLLECTION'

    elif category == 'Medium':
        return 'SCHEDULE COLLECTION'

    else:
        return 'NO IMMEDIATE COLLECTION'


df['Collection_Priority'] = df['Waste_Category'].apply(
    collection_priority
)

display(
    df[
        ['Bin_ID', 'Location', 'Waste_Level',
         'Waste_Category', 'Collection_Priority']
    ].head(15)
)

In [ ]:
priority_counts = df['Collection_Priority'].value_counts()

plt.figure(figsize=(8, 5))

priority_counts.plot(kind='bar')

plt.title('Garbage Collection Priority Summary')
plt.xlabel('Collection Priority')
plt.ylabel('Number of Bins')

plt.xticks(rotation=15)

plt.show()

In [ ]:
new_bin = pd.DataFrame({
    'Location': ['Market'],
    'Day': ['Saturday'],
    'Temperature': [35],
    'Population': [1200]
})

# Encode the categorical values
new_bin['Location'] = le_location.transform(new_bin['Location'])
new_bin['Day'] = le_day.transform(new_bin['Day'])

# Predict category
prediction = model.predict(new_bin)[0]

# Generate recommendation
priority = collection_priority(prediction)

print("SMART WASTE MANAGEMENT SYSTEM")
print("--------------------------------")
print("Location: Market")
print("Day: Saturday")
print("Temperature: 35°C")
print("Population: 1200")

print("\nPredicted Waste Category:", prediction)
print("Collection Priority:", priority)

In [ ]:
import joblib

joblib.dump(model, 'waste_classifier.pkl')

joblib.dump(le_location, 'location_encoder.pkl')
joblib.dump(le_day, 'day_encoder.pkl')

print("Models saved successfully!")